In [1]:
from llama_index.core.indices.multi_modal.base import MultiModalVectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import SimpleDirectoryReader, StorageContext
from llama_index.embeddings.clip import ClipEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import PromptTemplate
from llama_index.multi_modal_llms.ollama import OllamaMultiModal
from llama_index.llms.ollama import Ollama
from dotenv import load_dotenv
import os 
import gradio as gr

_ = load_dotenv(".env")


def load_documents():
    """Load the context images and text into ImageDocument and Documents"""
    # context images
    image_path = "../datasets_chatbot/images"
    image_documents = SimpleDirectoryReader(image_path).load_data()
    

    # context text
    text_path = "../datasets_chatbot/descriptions"
    text_documents = SimpleDirectoryReader(text_path).load_data()

    return image_documents, text_documents

def create_multimodal_index(image_documents, text_documents):

    # cretate multimodal embed model
    embeded_model = ClipEmbedding()

    # Define the sentence splitter
    node_parser = SentenceSplitter.from_defaults()
    image_nodes = node_parser.get_nodes_from_documents(image_documents)
    text_nodes = node_parser.get_nodes_from_documents(text_documents)

    # Create multimodal index
    index = MultiModalVectorStoreIndex(
    nodes = image_nodes + text_nodes,
    image_embed_model=embeded_model,
    embed_model = embeded_model
    )

    return index


def multimodal_rag(index: MultiModalVectorStoreIndex, llm:str ='llava:13b',k:int=1):

    # Define the prompt
    prompt_template = (
    "Images of shoes are provided.\n"
    "---------------------\n"
    "{context}\n"
    "---------------------\n"
    "If the images provided cannot help in answering the query\n"
    "then respond that you are unable to answer the query. Otherwise,\n"
    "using only the context provided, and not prior knowledge,\n"
    "provide an answer to the query."
    "Query: {query}\n"
    "Answer: "
    )

    prompt = PromptTemplate(prompt_template)

    # Instantiate the Ollama MultiModal LLM
    mm_model = OllamaMultiModal(model=llm, request_timeout=60.0)

    # Define rag query engine (Motor de Búsqueda)
    rag_engine = index.as_query_engine(
    llm = mm_model,
    prompt = prompt
    )

    # Retrieve settings. Finde the most relevant document
    rag_engine.retriever.image_similarity_top_k = 1
    rag_engine.retriever.similarity_top_k = k

    return rag_engine


def ingestion():

    # Load the context images and text
    image_documents, text_documents = load_documents()

    # Create index
    index = create_multimodal_index(image_documents, text_documents)

    return index


def rag_respond(query):
    global index
    global rag_engine
    response = rag_engine.query(query)
    return  gr.Image(response.metadata['image_nodes'][0].metadata['file_path']), response.response

def interface_chatbot():
    demo = gr.Interface(
    fn=rag_respond, 
    title="Multimodal Chatbot",
    inputs=[gr.Textbox()],   
    outputs=[gr.Image(), gr.Textbox()])
    demo.launch()
   

c:\Users\emada\miniconda3\envs\ia_mm_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Crear el indice multimodal con clip

In [2]:
index = ingestion()

* Busqueda por similitud

In [3]:
#Busqueda por similitud
retriever =index.as_retriever()
retriever.image_similarity_top_k = 1
retriever.similarity_top_k = 1

result = retriever.retrieve("Sneakers")
result

[NodeWithScore(node=TextNode(id_='da4b06c4-a9a2-4239-88f2-b531e407e6d4', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\5.txt', 'file_name': '5.txt', 'file_type': 'text/plain', 'file_size': 50, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='8a97902f-bca4-4a12-b757-c4267e32c4a6', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\5.txt', 'file_name': '5.txt', 'file_type': 'text/plain', '

In [5]:
#Busqueda por similitud
result2 = retriever.retrieve("Red sandals")
result2

[NodeWithScore(node=TextNode(id_='d27bc400-f450-45b5-9d94-4351bc7b3dee', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\4.txt', 'file_name': '4.txt', 'file_type': 'text/plain', 'file_size': 46, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='7d747ab0-c07c-4834-aaba-3d33788ccac4', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\4.txt', 'file_name': '4.txt', 'file_type': 'text/plain', '

In [6]:
#Busqueda por similitud
result3 = retriever.retrieve("denim boots")
result3

[NodeWithScore(node=TextNode(id_='cd57b0c0-755a-4e6b-bf6a-56ce20987cce', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\1.txt', 'file_name': '1.txt', 'file_type': 'text/plain', 'file_size': 54, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ba766b0b-c57c-4684-8721-7a95a62a023b', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\1.txt', 'file_name': '1.txt', 'file_type': 'text/plain', '

In [7]:
#Busqueda por similitud
result4 = retriever.retrieve("black floral boots")
result4

[NodeWithScore(node=TextNode(id_='f1f811be-e53d-46d4-8b69-5d4313c57ec9', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\2.txt', 'file_name': '2.txt', 'file_type': 'text/plain', 'file_size': 56, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='423f48d1-fad0-407c-9d39-cea6106eac57', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\2.txt', 'file_name': '2.txt', 'file_type': 'text/plain', '

In [8]:
#Busqueda por similitud
result5 = retriever.retrieve("black and white boots")
result5

[NodeWithScore(node=TextNode(id_='7a13590d-1112-48cd-88a6-44f0f1df49cc', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\3.txt', 'file_name': '3.txt', 'file_type': 'text/plain', 'file_size': 51, 'creation_date': '2025-01-07', 'last_modified_date': '2025-01-07'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='1733b338-f002-41d5-8b78-1258a9a0416d', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\descriptions\\3.txt', 'file_name': '3.txt', 'file_type': 'text/plain', '

In [42]:
#Busqueda por similitud
result6 = retriever.text_to_image_retrieve("black and white boots")
result6

[NodeWithScore(node=ImageNode(id_='af927831-a2a4-495d-bb0a-dc93346db318', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\3.jpg', 'file_name': '3.jpg', 'file_type': 'image/jpeg', 'file_size': 103383, 'creation_date': '2025-01-07', 'last_modified_date': '2024-12-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='81258254-434e-43f7-bbfe-1a5e5265b74a', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\3.jpg', 'file_name': '3.jpg', 'file_type': 'image/jpeg', 'file_si

In [46]:
#img_url = "..\\tests\\images\\12.jpg"
img_url = "..\\datasets_chatbot\\images\\1.jpg"
result7 = retriever.image_to_image_retrieve(img_url)
result7

[NodeWithScore(node=ImageNode(id_='b097c12e-fe9a-4e5c-add9-991d1c4e2025', embedding=None, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\1.jpg', 'file_name': '1.jpg', 'file_type': 'image/jpeg', 'file_size': 134819, 'creation_date': '2025-01-07', 'last_modified_date': '2024-12-04'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='b6aeab6e-43ed-4b08-8971-5bdba76fde1d', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'file_path': 'c:\\Users\\emada\\Desktop\\ia_workspace\\gcda_tfg\\poc_multimod_chatbot_chroma\\notebooks\\..\\datasets_chatbot\\images\\1.jpg', 'file_name': '1.jpg', 'file_type': 'image/jpeg', 'file_si

In [80]:
def similarity_search(query,image_path):
    """
    Search for similar images or text
    Args:
        query (str): Query text
        image_path (str): Image path
    """
    global index
    # Convert index as retriever
    retriever =index.as_retriever()
    retriever.image_similarity_top_k = 1
    retriever.similarity_top_k = 1
    # If query is not empty, conducts text-to-image search
    if query:
        # text to image, text search
        relevant_docs = retriever.retrieve(query)
        node_text = relevant_docs[0]
        node_img = relevant_docs[1]
        return  gr.Image(node_img.metadata['file_path']), gr.Textbox(node_text.text)
    else:
        # Otherwise, conducts the image to image search
        relevant_docs = retriever.image_to_image_retrieve(image_path)
        node_img = relevant_docs[0]

        # As we don't have the text, we have to read the text file manually
        image_path_found = node_img.metadata['file_path']
        # Replace from the image_path found the images dir by the descriptions dir and the extension
        text_path = image_path_found.replace("images","descriptions").replace("jpg","txt")
        with open(text_path, "r") as f:
            content_text = f.read()
        return  gr.Image(image_path_found), gr.Textbox(content_text)
    


In [81]:
similarity_search("black and white boots","")

(<gradio.components.image.Image at 0x26730a32450>,
 <gradio.components.textbox.Textbox at 0x2672f9c7da0>)

In [82]:
similarity_search("","..\\datasets_chatbot\\images\\1.jpg")

(<gradio.components.image.Image at 0x26731436780>,
 <gradio.components.textbox.Textbox at 0x2672f9be5a0>)

In [83]:
def interface_semantic_search():
    demo = gr.Interface(
    fn=similarity_search, 
    title="Multimodal Search Chatbot with Chroma and llamaIndex",
    inputs=[gr.Textbox(), gr.Image(type="filepath")],   
    outputs=[gr.Image(), gr.Textbox()])
    demo.launch()


In [84]:
interface_semantic_search()

Running on local URL:  http://127.0.0.1:7871

To create a public link, set `share=True` in `launch()`.


In [ ]:
#black and white boots
#red sandals
#sneakers
